**Navigation** : [Index](README.md) | [Suivant >>](10b-SemanticKernel-NotebookMaker-batch-parameterized.ipynb)

# ?? Conception Automatique de Notebook par Agents IA

Bienvenue dans ce d?monstrateur d'intelligence collective?!  
Trois agents sp?cialis?s collaborent pour d?velopper et valider votre notebook?:

- **???? Admin** : ?labore et finalise les sp?cifications dans les cellules Markdown.
- **?? Codeur** : impl?mente le code dans les cellules de type *code*.
- **?? Relecteur** : v?rifie le travail du codeur et valide (ou non) ses modifications.

## Comment d?marrer

1. Configurez votre t?che ci-dessous ?  
2. Lancez l'orchestration avec le bouton ??Play?? ou ??Ex?cuter??  
3. Observez la collaboration en direct?: chaque agent intervient ? tour de r?le  
4. Les cellules suivantes s?ex?cuteront automatiquement apr?s validation

**Conseil** : Dans votre explorateur de fichiers, ouvrez le notebook cible (nomm? "Notebook-Generated.ipynb" si vous avez choisi une t?che, ou bien le notebook t?l?vers?) pour voir ses cellules ?voluer en temps r?el?!


## Configuration du Projet

Dans cette section, plusieurs modes vous sont propos?s pour d?finir la t?che ? r?aliser :

- **Al?atoire** : choisit une t?che de mani?re al?atoire parmi une liste pr??tablie.  
- **Biblioth?que** : vous s?lectionnez la t?che d?sir?e dans un menu d?roulant.  
- **Personnalis?** : vous d?crivez librement votre t?che.  
- **Importer** : vous t?l?versez votre propre notebook `.ipynb`.

Cliquez sur **Valider** pour confirmer votre choix. Les cellules suivantes prendront automatiquement en compte cette configuration.


In [ ]:
# Cellule Code : Installation des d?pendances pour l'interface utilisateur
# ----------------------------------------------------------------------
# Ex?cutez cette cellule EN PREMIER, puis red?marrez le noyau si n?cessaire
# avant d'ex?cuter la cellule suivante.

print("D?pendances UI install?es. Red?marrez le noyau si vous rencontrez des probl?mes d'affichage des widgets.")

In [ ]:
# Parameters cell for Papermill execution
# Tagged with "parameters" for Papermill to inject values

# Notebook topic (the main subject/task for the notebook)
notebook_topic = "Simple notebook creation"  # Valeur par d?faut

# Notebook complexity level
notebook_complexity = "simple"  # simple|complex

# Additional requirements or specifications
additional_requirements = ""

# Target framework for the notebook
target_framework = "python"  # python|dotnet|hybrid

# Skip interactive widgets (for batch execution)
skip_widgets = True

# Legacy compatibility mapping
task_description = notebook_topic  # Map to existing logic
config_mode = 2  # Force Custom mode to use task_description


### Param?tres d'ex?cution Papermill

La cellule pr?c?dente d?finit les **param?tres par defaut** du notebook. Ces variables (`notebook_topic`, `notebook_complexity`, `target_framework`, `skip_widgets`, `config_mode`) peuvent etre surcharges par Papermill lors d'une ex?cution en batch, permettant ainsi de piloter le comportement du notebook sans modifier son code source.

**Points cles sur les param?tres** :

| Param?tre | Valeur par defaut | R?le |
|-----------|-------------------|------|
| `notebook_topic` | "Simple notebook creation" | Sujet principal du notebook a generer |
| `skip_widgets` | `True` | Desactive l'UI interactive (mode batch) |
| `config_mode` | 2 | S?lection du mode de configuration (0=Aleatoire, 1=Bibliotheque, 2=Personnalise) |

Les cellules suivantes definissent le mode batch et verifient que toutes les dependances necessaires sont disponibles avant de lancer la configuration du projet.

In [ ]:
# Parameters
BATCH_MODE = True


In [ ]:
# Verification des dependances (import guards)
try:
    from semantic_kernel import Kernel
    semantic_kernel_AVAILABLE = True
except ImportError:
    semantic_kernel_AVAILABLE = False
    print(f'WARNING: semantic-kernel non installe. Installez avec: pip install semantic-kernel')

try:
    import nbformat
    nbformat_AVAILABLE = True
except ImportError:
    nbformat_AVAILABLE = False
    print(f'WARNING: nbformat non installe. Installez avec: pip install nbformat')

try:
    import papermill as pm
    papermill_AVAILABLE = True
except ImportError:
    papermill_AVAILABLE = False
    print(f'WARNING: papermill non installe. Installez avec: pip install papermill')

try:
    from dotenv import load_dotenv
    dotenv_AVAILABLE = True
except ImportError:
    dotenv_AVAILABLE = False
    print(f'WARNING: python-dotenv non installe. Installez avec: pip install python-dotenv')

try:
    import ipywidgets as widgets
    ipywidgets_AVAILABLE = True
except ImportError:
    ipywidgets_AVAILABLE = False
    print(f'WARNING: ipywidgets non installe. Installez avec: pip install ipywidgets')


# %% Configuration du projet - Batch Mode

import time
import random

import ipywidgets as widgets
from IPython.display import display
from jupyter_ui_poll import ui_events

# ---- T?ches pr?d?finies propos?es ----
POSSIBLE_TASKS = [
    "Cr?er un notebook Python qui g?n?re al?atoirement un DataFrame de ventes mensuelles (12 mois), affiche des graphiques d'?volution et exporte un rapport PDF.",
    "Cr?er un notebook Python qui cr?e un dossier local avec quelques fichiers, puis compresse ce dossier en ZIP, et v?rifie la taille et l'int?grit? apr?s d?compression.",
    "Cr?er un notebook Python qui impl?mente un mini jeu console (Snake ou Pong) en mode 'demo' et se termine apr?s un certain nombre de 'ticks'.",
    "Cr?er un notebook Python utilisant openpyxl (ou xlsxwriter) pour g?n?rer deux fichiers Excel puis les fusionner avec un r?sum? global.",
    "Cr?er un notebook Python qui t?l?charge un flux RSS public (p.ex. CNN), stocke les titres dans un CSV, puis g?n?re un nuage de mots (WordCloud).",
    "Cr?er un notebook Python qui requ?te DBpedia (SPARQL) et affiche un graphique final (Plotly).",
    "Cr?er un notebook Python qui charge le dataset Titanic depuis une URL, effectue une analyse basique et un court mod?le de classification.",
    "Construire un notebook scikit-learn sur le dataset IRIS et r?aliser un court mod?le de classification."
]

# ---- Stockage de la configuration ----
config = {
    'mode': None,
    'task_description': None,
    'uploaded_file': None
}

# Flag qui indique si la config est pr?te
config_ready = False

# ---- MODE BATCH : Configuration automatique via param?tres ----
if skip_widgets:
    print("?? MODE BATCH ACTIV? - Configuration automatique sans UI")
    config['mode'] = config_mode
    
    if config_mode == 0:  # Al?atoire
        config['task_description'] = random.choice(POSSIBLE_TASKS)
        print(f"Mode: Al?atoire")
    elif config_mode == 1:  # Biblioth?que (utilise la premi?re t?che par d?faut)
        config['task_description'] = POSSIBLE_TASKS[0]
        print(f"Mode: Biblioth?que")
    elif config_mode == 2:  # Personnalis?
        config['task_description'] = task_description
        print(f"Mode: Personnalis?")
    elif config_mode == 3:  # Upload (non support? en batch)
        print("?? Mode Upload non support? en mode batch, utilisation mode Al?atoire")
        config['mode'] = 0
        config['task_description'] = random.choice(POSSIBLE_TASKS)
    
    print(f"T?che s?lectionn?e: {config['task_description'][:100]}...")
    config_ready = True

# ---- MODE INTERACTIF : UI avec widgets (code original) ----
else:
    # ---- Widgets de configuration ----
    task_selector = widgets.Dropdown(
        options=POSSIBLE_TASKS, 
        description='T?che :', 
        style={'description_width': 'initial'}
    )
    custom_task = widgets.Textarea(
        placeholder="D?crivez votre projet en d?tail...", 
        layout={'width': '90%', 'height': '120px'}
    )
    uploader    = widgets.FileUpload(accept='.ipynb', multiple=False)
    submit_btn  = widgets.Button(description="Valider", button_style='success', icon='rocket')

    tabs = widgets.Tab()
    tabs.children = [
        widgets.VBox([widgets.HTML("<i>Une t?che al?atoire sera g?n?r?e</i>")]),
        widgets.VBox([widgets.Label("Choisissez une t?che type :"), task_selector]),
        widgets.VBox([widgets.Label("?crivez vos instructions :"), custom_task]),
        widgets.VBox([widgets.Label("Uploader votre notebook :"), uploader])
    ]
    tabs.set_title(0, '?? Al?atoire')
    tabs.set_title(1, '?? Biblioth?que')
    tabs.set_title(2, '? Personnalis?')
    tabs.set_title(3, '?? Importer')

    def on_submit(_):
        """Callback d?clench? au clic du bouton."""
        global config_ready
        try:
            config['mode'] = tabs.selected_index
            
            if config['mode'] == 3:
                if uploader.value:
                    config['uploaded_file'] = uploader.value[0]
            else:
                if config['mode'] == 0:
                    config['task_description'] = random.choice(POSSIBLE_TASKS)
                elif config['mode'] == 1:
                    config['task_description'] = task_selector.value
                elif config['mode'] == 2:
                    config['task_description'] = custom_task.value

            submit_btn.disabled = True
            print("Configuration valid?e !")
            
        except Exception as e:
            print(f"Erreur pendant la configuration : {e}")

        config_ready = True

    submit_btn.on_click(on_submit)

    # ---- Affichage ----
    display(widgets.HTML("<h3>?? Configuration du Projet</h3>"))
    display(tabs)
    display(submit_btn)

    # ---- Boucle bloquante synchrone ----
    print("En attente du clic sur Valider...")

    with ui_events() as poll:
        while not config_ready:
            poll(10)
            time.sleep(0.1)

print("? Config termin?e, vous pouvez ex?cuter les cellules suivantes !")


## Configuration du LLM (.env)

Dans cette section, nous allons :

1. V?rifier si un fichier `.env` est pr?sent (et d?j? configur?) ou non.  
2. Vous proposer une interface pour saisir ou rappeler :  
   - La cl? d?API (`OPENAI_API_KEY`),  
   - L?URL d?un endpoint compatible OpenAI (`OPENAI_BASE_URL`),  
   - Le mod?le ? utiliser (`OPENAI_CHAT_MODEL_ID`).  
3. Mettre ? jour ou cr?er le fichier `.env` une fois la configuration valid?e.

Les cellules ult?rieures se baseront sur ces informations pour orchestrer les agents.


In [ ]:

import os
import time
import requests

from dotenv import load_dotenv
import ipywidgets as widgets
from IPython.display import display
from jupyter_ui_poll import ui_events

# -------------------------------------------------------------------------
# Fonctions utilitaires
# -------------------------------------------------------------------------
def list_models(api_base, api_key):
    """Retourne un dict avec la liste des mod?les ou un champ 'error'."""
    url = f"{api_base}/models"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    try:
        resp = requests.get(url, headers=headers, timeout=20)
        if resp.status_code == 200:
            return resp.json()
        else:
            return {"error": f"status={resp.status_code}", "text": resp.text}
    except Exception as e:
        return {"error": str(e)}

# -------------------------------------------------------------------------
# Lecture .env
# -------------------------------------------------------------------------
load_dotenv("../.env")

sd_fake = "sk-proj-1234567890"
openai_api_key       = os.getenv("OPENAI_API_KEY", sd_fake).strip()
openai_base_url      = os.getenv("OPENAI_BASE_URL", "").strip()
openai_chat_model_id = os.getenv("OPENAI_CHAT_MODEL_ID", "gpt-4o-mini").strip()

already_configured = (
    openai_api_key != sd_fake
    or (openai_base_url not in ["", "https://api.openai.com/v1"])
)

# Flag indiquant quand la config est OK
env_config_ready = False

# ---- MODE BATCH : Lecture directe du .env ----
if skip_widgets:
    print("?? MODE BATCH ACTIV? - Lecture configuration depuis .env")
    
    if already_configured:
        print("? Configuration LLM d?tect?e dans .env :")
        print(f"   - OPENAI_API_KEY       = {'configuree' if openai_api_key and openai_api_key != sd_fake else '(non configuree)'}")
        print(f"   - OPENAI_BASE_URL      = {openai_base_url or '(API officielle)'}")
        print(f"   - OPENAI_CHAT_MODEL_ID = {openai_chat_model_id}")
        env_config_ready = True
    else:
        print("?? Configuration LLM non d?tect?e dans .env")
        print("Veuillez configurer les variables d'environnement :")
        print("  - OPENAI_API_KEY")
        print("  - OPENAI_BASE_URL (optionnel)")
        print("  - OPENAI_CHAT_MODEL_ID")
        raise ValueError("Configuration LLM manquante pour mode batch")

# ---- MODE INTERACTIF : UI avec widgets (code original) ----
else:
    # -------------------------------------------------------------------------
    # Widgets 
    # -------------------------------------------------------------------------
    message_output = widgets.Output()

    api_key_input = widgets.Text(
        value=openai_api_key if already_configured else "",
        placeholder=f"Ex: {sd_fake}",
        description="Cl? API :",
        layout={'width': '80%'}
    )

    base_url_input = widgets.Text(
        value=openai_base_url if already_configured else "https://api.openai.com/v1",
        placeholder="ex: https://api.my-llm.com/v1",
        description="Endpoint :",
        layout={'width': '80%'}
    )

    model_dropdown = widgets.Dropdown(
        options=[],
        description="Mod?le :",
        layout={'width': '80%', 'display': 'none'}
    )

    list_models_btn = widgets.Button(
        description="Lister mod?les",
        button_style='info',
        icon='search'
    )

    validate_llm_btn = widgets.Button(
        description="Enregistrer .env",
        button_style='success',
        icon='save'
    )

    ui_box = widgets.VBox([
        api_key_input,
        base_url_input,
        model_dropdown,
        widgets.HBox([list_models_btn, validate_llm_btn])
    ])

    # -------------------------------------------------------------------------
    # Callbacks
    # -------------------------------------------------------------------------
    def on_list_models_click(_):
        new_base_url = base_url_input.value.strip()
        new_api_key  = api_key_input.value.strip()
        with message_output:
            message_output.clear_output()
            if not new_base_url or not new_api_key:
                print("?? Veuillez saisir un Endpoint et une cl? API avant de lister les mod?les.")
                return

            info = list_models(new_base_url, new_api_key)
            if "error" in info:
                print(f"Erreur /models: {info['error']} - {info.get('text','')}")
            else:
                data_list = info.get("data", [])
                if not data_list:
                    print("Aucun mod?le n'a ?t? retourn? (data=[]).")
                else:
                    model_ids = [m.get("id", "(inconnu)") for m in data_list]
                    model_dropdown.options = model_ids

                    if openai_chat_model_id in model_ids:
                        model_dropdown.value = openai_chat_model_id
                    else:
                        model_dropdown.value = model_ids[0]

                    model_dropdown.layout.display = 'block'
                    print(f"? {len(model_ids)} mod?le(s) trouv?(s).")

    def on_validate_llm_click(_):
        global env_config_ready
        new_api_key  = api_key_input.value.strip() or "sk-fake"
        new_base_url = base_url_input.value.strip()

        chosen_model = "gpt-3.5-turbo"
        if model_dropdown.options and (model_dropdown.layout.display != 'none'):
            chosen_model = model_dropdown.value.strip()

        with message_output:
            message_output.clear_output()
            try:
                with open('.env', 'w', encoding='utf-8') as f:
                    f.write(f"OPENAI_API_KEY={new_api_key}\n")
                    f.write(f"OPENAI_BASE_URL={new_base_url}\n")
                    f.write(f"OPENAI_CHAT_MODEL_ID={chosen_model}\n")

                print("? Fichier .env cr??/mis ? jour avec :")
                print(f"   - OPENAI_API_KEY       = {'configuree' if new_api_key and new_api_key != 'sk-fake' else '(non configuree)'}")
                print(f"   - OPENAI_BASE_URL      = {new_base_url or '(API OpenAI officiel)'}")
                print(f"   - OPENAI_CHAT_MODEL_ID = {chosen_model}")

                env_config_ready = True

            except Exception as e:
                print(f"? Erreur lors de l'?criture du fichier .env: {str(e)}")

    # -------------------------------------------------------------------------
    # Suppression des anciens callbacks
    # -------------------------------------------------------------------------
    list_models_btn._click_handlers.callbacks = []
    validate_llm_btn._click_handlers.callbacks = []

    list_models_btn.on_click(on_list_models_click)
    validate_llm_btn.on_click(on_validate_llm_click)

    # -------------------------------------------------------------------------
    # Affichage 
    # -------------------------------------------------------------------------
    if already_configured:
        print("? Configuration LLM d?tect?e dans .env :")
        print(f"   - OPENAI_API_KEY       = {'configuree' if openai_api_key and openai_api_key != sd_fake else '(non configuree)'}")
        print(f"   - OPENAI_BASE_URL      = {openai_base_url or '(API officielle)'}")
        print(f"   - OPENAI_CHAT_MODEL_ID = {openai_chat_model_id}")
        print("Aucune saisie suppl?mentaire n'est requise.")
        env_config_ready = True
    else:
        print("Veuillez :\n1) Saisir votre Endpoint et cl? API")
        print("2) Cliquer sur [Lister mod?les] (pour un endpoint custom)")
        print("3) Cliquer sur [Enregistrer .env] pour finaliser la configuration")
        display(ui_box)
        display(message_output)
        
        # ---- Boucle bloquante: attend le clic sur "Enregistrer .env" ----
        with ui_events() as poll:
            while not env_config_ready:
                poll(10)
                time.sleep(0.1)

        ui_box.layout.display = 'none'

    print("? Configuration LLM termin?e, vous pouvez ex?cuter la suite !")


## ? D?marrage du Processus

La configuration ?tant termin?e, les ?tapes suivantes vont se lancer :

1. **Installation des d?pendances** : Nous v?rifions et installons papermill, nbformat, semantic-kernel, etc.  
2. **Gestion d??tat** : nous utilisons la classe `NotebookState` pour piloter le cycle de vie du notebook.  
3. **Plugins** : chaque agent aura un *plugin* lui permettant de lire ou modifier le contenu du notebook.  
4. **Strat?gies d'orchestration** : nous d?finissons quelles actions lancer en fonction de l??tat (`specified`, `implemented`, `tested`, `validated`).  
5. **Agents** : d?finition et configuration des trois agents (`AdminAgent`, `CoderAgent`, `ReviewerAgent`).  
6. **Conversation multi-agents** : la conversation s?encha?ne jusqu?? validation du notebook (ou d?passement du nombre maximal d?it?rations).

**Objectif final** : Obtenir un notebook enti?rement fonctionnel et valid?.


## 1. Installation des d?pendances

Nous installons ici les biblioth?ques n?cessaires :

- **papermill** : pour ex?cuter des notebooks et injecter des variables,  
- **nbformat** : pour manipuler la structure interne d?un notebook,  
- **semantic-kernel** : pour orchestrer la collaboration entre plusieurs agents (LLM).


In [ ]:
# Cellule Code : Installation des packages requis
# ------------------------------------------------
# Nous installons ici les packages indispensables au pipeline.

print("Installation termin?e. Si n?cessaire, red?marrez le kernel pour activer les nouveaux packages.")


## 2. Import des biblioth?ques et configuration

Nous importons :

- Les biblioth?ques standard (os, json, logging, etc.).  
- Les biblioth?ques *notebook* (nbformat, papermill).  
- `semantic-kernel` pour la gestion de nos agents conversationnels.  
- Un logger color? pour am?liorer la lisibilit? et le suivi de l?ex?cution.


In [ ]:
# Cellule Code : Imports et configuration du logger
# -------------------------------------------------
import os
import json
import hashlib
import logging
import nbformat
import papermill as pm
import random
from datetime import datetime

# Imports li?s ? Semantic Kernel
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.functions import kernel_function
from semantic_kernel.agents.strategies.selection.selection_strategy import SelectionStrategy
from semantic_kernel.agents.strategies.termination.termination_strategy import TerminationStrategy
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.functions.kernel_arguments import KernelArguments

class ColorFormatter(logging.Formatter):
    """
    Un formatter color? pour rendre les logs plus lisibles.
    """
    colors = {
        'DEBUG': '\033[94m',
        'INFO': '\033[92m',
        'WARNING': '\033[93m',
        'ERROR': '\033[91m',
        'CRITICAL': '\033[91m\033[1m'
    }
    reset = '\033[0m'

    def format(self, record: logging.LogRecord) -> str:
        msg = super().format(record)
        return f"{self.colors.get(record.levelname, '')}{msg}{self.reset}"

logger = logging.getLogger("Orchestration")
logger.setLevel(logging.DEBUG)  # Peut ?tre param?tr? via .env ou variable

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.DEBUG)
    formatter = ColorFormatter(
        fmt="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
        datefmt="%H:%M:%S"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Configuration initiale termin?e.")


## 3. Classe `NotebookState` : gestion de l??tat du notebook

Cette classe regroupe :

- La lecture et l??criture du fichier `.ipynb`,  
- L?ex?cution via Papermill pour d?tecter d??ventuelles erreurs,  
- Les transitions d??tats du notebook : `specified`, `implemented`, `tested`, `validated`,  
- Les mises ? jour cibl?es d?une cellule d?termin?e (recherche par contenu).

Elle centralise toutes les op?rations afin que chaque agent puisse y acc?der.


In [ ]:
class NotebookState:
    """
    G?re le statut et le contenu d'un notebook.
    
    Les ?tats possibles sont :
      - 'specified'
      - 'implemented'
      - 'tested'
      - 'validated'

    Attributes:
        notebook_path (str): Chemin du fichier notebook (.ipynb).
        _cached_notebook (nbformat.NotebookNode): Le contenu du notebook stock? en m?moire.
        _status (str): L'?tat courant du notebook.
        _previous_states (list[str]): Historique simple des ?tats ant?rieurs.
    """

    VALID_STATES = ["specified", "implemented", "tested", "validated"]

    def __init__(self, notebook_path: str) -> None:
        self.notebook_path = notebook_path
        self._cached_notebook = None
        self._status = "specified"
        self._previous_states = []
        self._load_notebook()

    def _load_notebook(self) -> None:
        """Charge le notebook depuis le chemin sp?cifi?, en utilisant nbformat."""
        if not os.path.exists(self.notebook_path):
            raise FileNotFoundError(f"Notebook introuvable: {self.notebook_path}")

        with open(self.notebook_path, "r", encoding="utf-8") as f:
            self._cached_notebook = nbformat.read(f, as_version=4)

        logger.debug(
            f"[NotebookState] Charg? '{self.notebook_path}' avec {len(self._cached_notebook.cells)} cellules."
        )

    def save_notebook(self, path: str = "") -> None:
        """
        Enregistre le notebook sur disque (par d?faut au m?me chemin).
        """
        if not path:
            path = self.notebook_path
        try:
            with open(path, "w", encoding="utf-8") as f:
                nbformat.write(self._cached_notebook, f)
            logger.info(f"[NotebookState] Notebook sauvegard? sous {path}")
        except Exception as e:
            logger.error(f"[save_notebook] Erreur de sauvegarde: {e}")
            raise

    def get_notebook_json(self) -> str:
        """Retourne une repr?sentation JSON (str) du notebook actuellement en m?moire."""
        return json.dumps(self._cached_notebook, indent=2, ensure_ascii=False)

    def log_notebook_state(self, max_length: int = 10000) -> None:
        """Affiche dans les logs une partie du JSON pour diagnostic (tronqu?e si trop longue)."""
        notebook_json = self.get_notebook_json()
        snippet = notebook_json[:max_length]
        if len(notebook_json) > max_length:
            snippet += ">>> TRUNCATED <<<"
        logger.debug(f"[NotebookState] Current notebook state (truncated):\n{snippet}")

    def get_status(self) -> str:
        """Renvoie l'?tat courant du notebook."""
        return self._status

    def set_status(self, new_status: str) -> None:
        """
        Met ? jour l'?tat du notebook et log la transition.
        Ne fait rien si new_status est invalide.
        """
        if new_status not in self.VALID_STATES:
            logger.error(f"[NotebookState] ?tat invalide: {new_status}")
            return
        logger.info(f"[NotebookState] Passage de l'?tat {self._status} ? {new_status}")
        self._previous_states.append(self._status)
        self._status = new_status

    def reset_outputs(self) -> None:
        """
        Efface les sorties de toutes les cellules (execution_count, outputs).
        Utile avant r?-ex?cution si on veut repartir ? z?ro.
        """
        for cell in self._cached_notebook["cells"]:
            if "outputs" in cell:
                cell["outputs"] = []
            if "execution_count" in cell:
                cell["execution_count"] = None
        logger.debug("[NotebookState] Sorties r?initialis?es dans le notebook.")

    def execute_notebook(self) -> bool:
        """
        Ex?cute le notebook via Papermill et retourne True si tout se passe bien,
        ou False si une exception survient.
        """
        logger.info(f"[NotebookState] Ex?cution Papermill sur {self.notebook_path}.")
        self.save_notebook()  # Sauvegarde avant ex?cution

        success = True
        try:
            pm.execute_notebook(
                input_path=self.notebook_path,
                output_path=self.notebook_path,
                kernel_name="python3",
                progress_bar=False,
                log_output=True
            )
        except Exception as e:
            logger.error(f"[execute_notebook] Erreur lors de l'ex?cution: {e}")
            success = False
        finally:
            self._load_notebook()  # Recharger, car Papermill a peut-?tre modifi? le contenu
            self.save_notebook()
            logger.info("[NotebookState] Notebook mis ? jour apr?s ex?cution (avec sorties).")

        return success

    def update_cell(self, cell_index: int, new_source: str) -> None:
        """
        Met ? jour la source d?une cellule (index) et sauvegarde le notebook.
        """
        old_src = self._cached_notebook.cells[cell_index].source
        self._cached_notebook.cells[cell_index].source = new_source
        logger.info(f"[NotebookState] Mise ? jour de la cellule {cell_index}")
        logger.debug(f"[NotebookState] Ancien contenu:\n{old_src}")
        logger.debug(f"[NotebookState] Nouveau contenu:\n{new_source}")
        self.save_notebook()

    def find_cell_indices_by_content(self, content_pattern: str) -> list:
        """Retourne la liste des indices de cellules contenant `content_pattern`."""
        indices = []
        for i, c in enumerate(self._cached_notebook.cells):
            if content_pattern in c.source:
                indices.append(i)
        return indices

    def is_approved(self) -> bool:
        """Renvoie True si l'?tat du notebook est 'validated'."""
        return self._status == "validated"


### Exercice 1 : Ajout d'une m?thode de resume du notebook

La classe `NotebookState` permet de lire et modifier un notebook. Il manque une m?thode qui genere un resume textuel utile pour le diagnostic.

**Objectif** : Ajouter une m?thode `get_summary()` a `NotebookState` qui retourne un resume structure du notebook.

**Indices** :
- # ?tape 1 : Compter le nombre de cellules par type (code vs markdown)
- # ?tape 2 : Calculer la taille totale (en caract?res) des cellules
- # ?tape 3 : Identifier la cellule la plus longue et la plus courte
- # Indice : Utiliser `len(cell.source)` pour la taille et `cell.cell_type` pour le type

In [ ]:
def get_notebook_summary(notebook_state: NotebookState) -> dict:
    """
    TODO etudiant : Generer un resume structure du notebook.
    
    Returns:
        dict avec les cles: total_cells, code_cells, markdown_cells,
        total_chars, longest_cell_index, shortest_cell_index, current_status
    """
    # TODO etudiant : analyser notebook_state._cached_notebook.cells
    return None

# Test avec le notebook charge :
# summary = get_notebook_summary(notebook_state)
# print(f"Cellules: {summary['total_cells']} ({summary['code_cells']} code, {summary['markdown_cells']} markdown)")
print("Exercice a completer")

## 4. Cr?ation (ou chargement) du Notebook cible + test de validit?

Voici les ?tapes automatis?es :

1. Nous utilisons un template `Notebook-Template.ipynb`, ou bien le fichier `.ipynb` t?l?vers?,  
2. Nous injectons une description de t?che (si elle est choisie al?atoirement, pr?d?finie ou personnalis?e),  
3. Nous ex?cutons le notebook pour v?rifier qu?aucune erreur fatale n?appara?t,  
4. Nous validons l?int?grit? pour confirmer que tout est correctement initialis?.

Si tout se passe bien, la phase d?orchestration multi-agents peut commencer.


In [ ]:

# Cellule Code : Initialisation du notebook cibl? et injection de la t?che
# -----------------------------------------------------------------------
import shutil
import os
import requests

TEMPLATE_URL = "https://raw.githubusercontent.com/jsboige/CoursIA/refs/heads/main/MyIA.AI.Notebooks/GenAI/SemanticKernel/Notebook-Template.ipynb"
TEMPLATE_FILE = "Notebook-Template.ipynb"

def apply_task_description(notebook_state: NotebookState, task_description: str) -> bool:
    """
    Remplace le placeholder {{TASK_DESCRIPTION}} dans la premi?re cellule Markdown appropri?e.
    Retourne True si le placeholder a ?t? trouv? et remplac?, False sinon.
    """
    for idx, cell in enumerate(notebook_state._cached_notebook.cells):
        if "{{TASK_DESCRIPTION}}" in cell.source:
            new_source = cell.source.replace("{{TASK_DESCRIPTION}}", task_description)
            notebook_state.update_cell(idx, new_source)
            return True
    return False

DEST_NOTEBOOK = "Notebook-Generated.ipynb"

# Gestion diff?renci?e selon le mode s?lectionn?
if config['mode'] == 3:  # Mode upload
    if config['uploaded_file']:
        DEST_NOTEBOOK = config['uploaded_file'].name
        with open(DEST_NOTEBOOK, 'wb') as f:
            f.write(config['uploaded_file'].content)
        logger.info(f"Notebook upload? : {DEST_NOTEBOOK}")
else:  # Modes template (al?atoire, biblioth?que, personnalis?)
    if not os.path.exists(TEMPLATE_FILE):
        print(f"{TEMPLATE_FILE} introuvable, t?l?chargement depuis {TEMPLATE_URL}")
        try:
            response = requests.get(TEMPLATE_URL, timeout=10)
            response.raise_for_status()  # G?re les erreurs HTTP
            with open(TEMPLATE_FILE, "wb") as f:
                f.write(response.content)
            print(f"T?l?chargement termin?, fichier {TEMPLATE_FILE} cr??.")
        except Exception as e:
            print(f"?chec du t?l?chargement : {e}")
    else:
        print(f"Le fichier {TEMPLATE_FILE} existe d?j?, aucune action n?cessaire.")
    
    shutil.copy2(TEMPLATE_FILE, DEST_NOTEBOOK)
    logger.info(f"Cr?ation depuis le template : {TEMPLATE_FILE}")

# Instanciation du notebook state
notebook_state = NotebookState(DEST_NOTEBOOK)
notebook_state.log_notebook_state()

# Injection de la t?che pour les modes template
if config['mode'] != 3:
    changed = apply_task_description(notebook_state, config['task_description'])
    
    if changed:
        logger.info("Placeholder {{TASK_DESCRIPTION}} remplac? avec succ?s.")
        logger.info("Ex?cution du notebook pour validation initiale...")
        notebook_state.execute_notebook()
        logger.info("Notebook r?-ex?cut? apr?s injection de la t?che.")
        notebook_state.log_notebook_state()
    else:
        logger.warning("Aucun placeholder d?tect?. V?rifiez la cellule Markdown contenant {{TASK_DESCRIPTION}}.")
else:
    logger.info("Mode upload - Aucune injection de t?che n?cessaire")

## 5. Architecture de plugins : extension du NotebookState

Nous d?finissons des **plugins** pour manipuler `NotebookState` :

- **BaseNotebookPlugin** : expose en lecture le notebook (m?thode `get_notebook_content()`).  
- **CoderNotebookPlugin** : permet de modifier des cellules de code et de signaler la fin d?impl?mentation.  
- **ReviewerNotebookPlugin** : ex?cute le notebook et d?cide d?approuver ou de refuser.  
- **AdminNotebookPlugin** : finalise ou rejette le notebook, et peut ?diter les cellules Markdown.

Chaque plugin est un ensemble de fonctions d?cor?es (`@kernel_function`), utilisables par les agents via Semantic Kernel.


In [ ]:
# ================================
# Plugins actualis?s (avec logs)
# ================================
class BaseNotebookPlugin:
    """
    Plugin de base pour manipuler NotebookState.
    Fournit la m?thode get_notebook_content() 
    pour r?cup?rer le notebook en JSON.
    """

    def __init__(self, state: NotebookState) -> None:
        self.state = state
        self._get_content_counter = 0  # Pour logguer tous les 5 appels

    @kernel_function(
        name="get_notebook_content",
        description="Renvoie le notebook (format JSON) r?vis? actuellement."
    )
    def get_notebook_content(self) -> str:
        """
        Retourne la repr?sentation JSON du notebook.
        Loggue un extrait toutes les 5 demandes pour ?viter la surcharge.
        """
        self._get_content_counter += 1
        content = self.state.get_notebook_json()

        if (self._get_content_counter % 5) == 0:
            logger.info(f"[BaseNotebookPlugin] get_notebook_content() (appel n?{self._get_content_counter}) - log complet")
            self.state.log_notebook_state(max_length=10000)
        else:
            snippet = content[:200]
            snippet += "..." if len(content) > 200 else ""
            logger.info(f"[BaseNotebookPlugin] get_notebook_content() (appel n?{self._get_content_counter}) -> {snippet}")

        return content


class NotebookEditingMixin:
    """
    Mixin fournissant la fonction d'?dition de cellule (update_cell_anyway).
    Peut ?tre h?rit? par Coder ou Admin, qui ont tous deux besoin d'?diter.
    """

    def update_cell_anyway(self, pattern: str, new_source: str, cell_type: str = None) -> str:
        """
        Recherche la premi?re cellule contenant 'pattern' (dans le code ou markdown),
        puis remplace son contenu par 'new_source', si l'?tat le permet.
        """
        indices = []
        for idx, cell in enumerate(self.state._cached_notebook.cells):
            content_match = (pattern in cell.source)
            type_match = (cell_type is None) or (cell.cell_type == cell_type)
            if content_match and type_match:
                indices.append(idx)

        if not indices:
            return_message = f"Aucune cellule ({cell_type or 'tout type'}) ne contient '{pattern}'."
        elif len(indices) > 1:
            return_message = f"Plusieurs cellules ({cell_type or 'tout type'}) contiennent '{pattern}'."
        else:
            old_status = self.state.get_status()
            if old_status == "validated":
                return_message = "?dition impossible (notebook d?j? valid?)."
            else:
                self.state.update_cell(indices[0], new_source)
                return_message = f"Cellule {cell_type} contenant '{pattern}' mise ? jour."

        logger.info(f"[NotebookEditingMixin] update_cell_anyway -> {return_message}")
        return return_message


class CoderNotebookPlugin(BaseNotebookPlugin, NotebookEditingMixin):
    """
    Plugin pour l'agent 'Coder'.
    H?rite de BaseNotebookPlugin (lecture) et NotebookEditingMixin (?dition).
    """

    @kernel_function(
        name="update_cell_by_content",
        description="Modifie la premi?re cellule de Code contenant 'content_pattern'."
    )
    def update_cell_by_content(self, content_pattern: str, new_source: str) -> str:
        logger.info("[CoderNotebookPlugin] update_cell_by_content()")
        status = self.state.get_status()

        if status not in ["specified", "implemented"]:
            msg = f"Erreur: ?tat '{status}' => modifications bloqu?es."
        else:
            msg = self.update_cell_anyway(
                pattern=content_pattern,
                new_source=new_source,
                cell_type="code"
            )
        logger.info(f"[CoderNotebookPlugin] update_cell_by_content -> {msg}")
        return msg

    @kernel_function(
        name="finish_implementation",
        description="D?clare le notebook 'implemented' lorsque le code est pr?t."
    )
    def finish_implementation(self) -> str:
        logger.info("[CoderNotebookPlugin] finish_implementation()")
        status = self.state.get_status()

        if status == "specified":
            self.state.set_status("implemented")
            msg = "Le notebook passe ? l'?tat 'implemented'."
        elif status == "implemented":
            msg = "Le notebook est d?j? en ?tat 'implemented'."
        else:
            msg = f"Impossible de passer en 'implemented' depuis '{status}'."

        logger.info(f"[CoderNotebookPlugin] finish_implementation -> {msg}")
        return msg


class ReviewerNotebookPlugin(BaseNotebookPlugin):
    """
    Plugin pour l'agent 'Reviewer'. 
    Il ne peut pas ?diter le notebook, mais peut l'ex?cuter et approuver (ou non).
    """

    @kernel_function(
        name="validate_notebook",
        description="Ex?cute le notebook et approuve ou non (approve=True/False)."
    )
    def validate_notebook(self, approve: bool = True) -> str:
        logger.info(f"[ReviewerNotebookPlugin] validate_notebook(approve={approve})")
        status = self.state.get_status()

        if status != "implemented":
            msg = f"Le reviewer ne peut pas valider, ?tat actuel = '{status}'."
        else:
            success = self.state.execute_notebook()
            if not success:
                self.state.set_status("specified")
                msg = ("Erreur d'ex?cution dans le notebook (voir logs). "
                       "Retour ? l'?tat 'specified' pour corrections.")
            else:
                if approve:
                    self.state.set_status("tested")
                    msg = "Le reviewer approuve => ?tat 'tested'."
                else:
                    self.state.set_status("specified")
                    msg = "Le reviewer refuse => retour ? 'specified'."

        logger.info(f"[ReviewerNotebookPlugin] validate_notebook -> {msg}")
        return msg


class AdminNotebookPlugin(BaseNotebookPlugin, NotebookEditingMixin):
    """
    Plugin pour l'agent 'Admin'.
    Peut lire, ?diter et valider ou invalider le notebook.
    """

    @kernel_function(
        name="update_markdown_cell",
        description="Modifie la premi?re cellule MARKDOWN contenant 'content_pattern'."
    )
    def admin_edit_markdown_cell(self, content_pattern: str, new_source: str) -> str:
        logger.info("[AdminNotebookPlugin] admin_edit_markdown_cell()")
        msg = self.update_cell_anyway(
            pattern=content_pattern,
            new_source=new_source,
            cell_type="markdown"
        )
        # Apr?s ?dition, on repasse l'?tat ? 'specified'.
        self.state.set_status("specified")

        logger.info(f"[AdminNotebookPlugin] admin_edit_markdown_cell -> {msg}")
        return msg + " -> Revert ? 'specified'."

    @kernel_function(
        name="approve_notebook",
        description="Validation finale: admin_ok=True => 'validated', sinon 'specified'."
    )
    def approve_notebook(self, admin_ok: bool = True) -> str:
        logger.info(f"[AdminNotebookPlugin] approve_notebook(admin_ok={admin_ok})")
        status = self.state.get_status()

        if status != "tested":
            msg = f"Impossible d'approuver: l'?tat est '{status}' (attendu: 'tested')."
        else:
            if admin_ok:
                self.state.set_status("validated")
                msg = "Notebook valid? => ?tat 'validated'."
            else:
                self.state.set_status("specified")
                msg = "Admin refuse => retour ? 'specified'."

        logger.info(f"[AdminNotebookPlugin] approve_notebook -> {msg}")
        return msg


### Exercice 2 : Plugin de validation syntaxique

Le `ReviewerNotebookPlugin` execute le notebook via Papermill pour detecter les erreurs. Cependant, une validation syntaxique prealable pourrait eviter des ex?cutions inutiles.

**Objectif** : Cr?er une m?thode `check_syntax()` dans une classe `SyntaxCheckerPlugin` qui verifie la syntaxe Python des cellules de code sans les executer.

**Indices** :
- # ?tape 1 : Utiliser `ast.parse()` du module standard `ast` pour verifier la syntaxe
- # ?tape 2 : Iterer sur les cellules du notebook et tester uniquement les cellules de type "code"
- # ?tape 3 : Retourner un rapport listant les cellules avec erreurs (index + message d'erreur)
- # Indice : Capturer `SyntaxError` et `IndentationError` pour chaque cellule

In [ ]:
import ast

class SyntaxCheckerPlugin(BaseNotebookPlugin):
    """
    TODO etudiant : Plugin de validation syntaxique.
    """
    
    @kernel_function(
        name="check_syntax",
        description="Verifie la syntaxe Python de toutes les cellules de code."
    )
    def check_syntax(self) -> str:
        """
        TODO etudiant : Verifier la syntaxe de chaque cellule de code.
        Retourner un rapport avec les erreurs detectees.
        """
        # TODO etudiant : iterer sur self.state._cached_notebook.cells
        # et utiliser ast.parse() sur chaque cellule de type "code"
        return None

print("Exercice a completer")

## 6. Strat?gies d?orchestration

Deux strat?gies principales :

1. **ApprovedBasedTerminationStrategy**  
   - Met fin ? la conversation d?s que le notebook est valid?, ou si un nombre maximal d?it?rations est atteint.  

2. **NotebookAwareSelectionStrategy**  
   - S?lectionne l?agent en fonction de l??tat courant du notebook :  
     - `specified` ? **CoderAgent**  
     - `implemented` ? **ReviewerAgent**  
     - `tested` ? **AdminAgent**  
     - `validated` ? plus d?agent (arr?t de la conversation)  
   - Tient aussi compte de la premi?re intervention pour laisser l?Admin faire ses modifications initiales.


In [ ]:
from pydantic import PrivateAttr

class ApprovedBasedTerminationStrategy(TerminationStrategy):
    """
    Met fin ? la conversation si le notebook est valid? ou si 
    le nombre d'it?rations maximum est atteint.
    """
    _state: NotebookState = PrivateAttr()
    _max_steps: int = PrivateAttr(default=20)

    def __init__(self, state: NotebookState, max_steps: int = 20):
        super().__init__()
        self._state = state
        self._max_steps = max_steps
        self._current_step = 0

    async def should_agent_terminate(self, agent, history) -> bool:
        self._current_step += 1
        is_approved = self._state.is_approved()
        logger.debug(
            f"[TerminationStrategy] Step={self._current_step}/{self._max_steps}, IsApproved={is_approved}"
        )
        if is_approved:
            logger.info("[TerminationStrategy] Notebook approuv? => arr?t.")
            return True
        if self._current_step >= self._max_steps:
            logger.warning(f"[TerminationStrategy] max_steps={self._max_steps} atteint => arr?t.")
            return True
        return False


class NotebookAwareSelectionStrategy(SelectionStrategy):
    """
    S?lectionne quel agent doit parler en fonction de l'?tat du notebook.
    """
    def __init__(self, state: NotebookState):
        super().__init__()
        self._state = state
        self._has_first_agent_run = False

    def reset(self) -> None:
        self._has_first_agent_run = False

    async def select_agent(self, agents, history):
        current_status = self._state.get_status()
        logger.debug(
            f"[SelectionStrategy] nb_agents={len(agents)}, statut={current_status}, "
            f"first_run={not self._has_first_agent_run}"
        )

        if current_status == "validated":
            logger.info("[SelectionStrategy] Notebook d?j? valid? => fin.")
            return None

        # R?cup?rer les agents
        coder = next((a for a in agents if a.name == "CoderAgent"), None)
        reviewer = next((a for a in agents if a.name == "ReviewerAgent"), None)
        admin = next((a for a in agents if a.name == "AdminAgent"), None)

        # Cas particulier : premi?re intervention => Admin
        if not self._has_first_agent_run and current_status == "specified":
            if admin:
                self._has_first_agent_run = True
                logger.info("Premi?re intervention : AdminAgent (r?vision du Markdown).")
                return admin

        # Cas normal
        if current_status == "specified" and coder:
            to_return = coder
        elif current_status == "implemented" and reviewer:
            to_return = reviewer
        elif current_status == "tested" and admin:
            to_return = admin
        else:
            logger.warning(
                f"[SelectionStrategy] Aucun agent trouv? pour ?tat='{current_status}' => stop."
            )
            to_return = None

        if to_return and not self._has_first_agent_run:
            self._has_first_agent_run = True

        if to_return:
            logger.info(f"[SelectionStrategy] Agent s?lectionn? : {to_return.name}")
        return to_return


## 7. Cr?ation des 3 agents (Coder, Reviewer, Admin)

- Chaque agent dispose d?un `kernel` ind?pendant, reli? ? un plugin d?di?,  
- Le service ChatCompletion (OpenAI) ou un endpoint custom est configur? via le `.env`,  
- Nous activons le comportement ??Auto ? pour le choix et l?appel des fonctions,  
- Les r?les :  
  - **CoderAgent** : impl?mente les cellules de code,  
  - **ReviewerAgent** : ex?cute et valide (ou non) apr?s relecture,  
  - **AdminAgent** : valide, invalide, ou r??dite les sp?cifications dans le Markdown.


In [ ]:
# Cellule Code : Cr?ation des 3 agents et configuration avec prise en compte du .env
# ---------------------------------------------------------------------------------
from openai import AsyncOpenAI
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
import os
from dotenv import load_dotenv

load_dotenv("../.env")
openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url_from_env = os.getenv("OPENAI_BASE_URL", "").strip()
openai_chat_model_id = os.getenv("OPENAI_CHAT_MODEL_ID", "gpt-4o-mini")


def create_chat_completion_service(service_id: str = "default"):
    """
    Cr?e une instance de ChatCompletion (OpenAI) en fonction
    des variables d'environnement. G?re explicitement l'URL par d?faut pour ?viter
    les erreurs de protocole non support?.

    Variables .env utilis?es :
     - OPENAI_API_KEY       : Cl? API (obligatoire).
     - OPENAI_CHAT_MODEL_ID : Nom du mod?le (ex: "gpt-4o", "gpt-4o-mini").
     - OPENAI_BASE_URL      : URL personnalis?e (optionnelle). Si vide, utilise l'API OpenAI officielle.
    """

    if not openai_api_key:
        # Il est crucial d'avoir une cl? API
        logger.error("La variable d'environnement OPENAI_API_KEY n'est pas d?finie !")
        raise ValueError("OPENAI_API_KEY is not set in the environment.")

    # D?terminer l'URL de base finale ? utiliser
    if openai_base_url_from_env:
        # Un endpoint personnalis? est d?fini dans le .env
        final_base_url = openai_base_url_from_env
        logger.info(f"Utilisation d'un endpoint compatible OpenAI : {final_base_url}")
    else:
        # Pas d'endpoint personnalis?, utiliser l'API officielle avec URL explicite
        final_base_url = "https://api.openai.com/v1"
        logger.info(f"Utilisation du service OpenAI officiel (URL explicite: {final_base_url}).")

    # Cr?er un client AsyncOpenAI configur? avec l'URL finale et la cl?
    # Ce client sera pass? au service Semantic Kernel
    try:
        openai_async_client = AsyncOpenAI(
            api_key=openai_api_key,
            base_url=final_base_url
            # Vous pouvez ajouter d'autres param?tres ici si n?cessaire (ex: timeout)
        )
    except Exception as client_error:
        logger.error(f"Erreur lors de la cr?ation du client AsyncOpenAI pour base_url='{final_base_url}': {client_error}")
        raise

    # Instancier le service Semantic Kernel en lui passant le client pr?-configur?
    try:
        sk_service = OpenAIChatCompletion(
            service_id=service_id,
            ai_model_id=openai_chat_model_id,
            async_client=openai_async_client # Important: passer le client configur?
            # api_key=... n'est pas n?cessaire ici car g?r? par le client
        )
        logger.debug(f"Service Semantic Kernel '{service_id}' cr?? pour le mod?le '{openai_chat_model_id}' pointant vers '{final_base_url}'")
        return sk_service
    except Exception as sk_service_error:
        logger.error(f"Erreur lors de la cr?ation du service OpenAIChatCompletion de Semantic Kernel: {sk_service_error}")
        raise

def create_kernel_for_agent(agent_id: str, plugin_instance) -> Kernel:
    """
    Instancie un Kernel SemanticKernel pour l'agent donn?,
    en y ajoutant le plugin associ? (Coder, Reviewer ou Admin).
    Utilise create_chat_completion_service() pour choisir le backend OpenAI
    (officiel ou endpoint custom) en fonction du .env.
    """
    k = Kernel()

    # Instancier le service de ChatCompletion en fonction du .env
    chat_service = create_chat_completion_service(service_id="default")

    # Ajouter ce service au kernel
    k.add_service(chat_service)

    # Ajouter le plugin de l'agent (Coder, Reviewer, Admin)
    k.add_plugin(plugin_instance, plugin_name=f"{agent_id}_plugin")

    return k

# Instancier les plugins (bas?s sur NotebookState)
coder_plugin = CoderNotebookPlugin(notebook_state)
reviewer_plugin = ReviewerNotebookPlugin(notebook_state)
admin_plugin = AdminNotebookPlugin(notebook_state)

# Cr?er un kernel d?di? par agent (en tenant compte du .env)
coder_kernel = create_kernel_for_agent("coder_kernel", coder_plugin)
reviewer_kernel = create_kernel_for_agent("reviewer_kernel", reviewer_plugin)
admin_kernel = create_kernel_for_agent("admin_kernel", admin_plugin)

# Configurer l'auto-appel de fonctions
coder_settings = coder_kernel.get_prompt_execution_settings_from_service_id("default")
coder_settings.function_choice_behavior = FunctionChoiceBehavior.Auto()

reviewer_settings = reviewer_kernel.get_prompt_execution_settings_from_service_id("default")
reviewer_settings.function_choice_behavior = FunctionChoiceBehavior.Auto()

admin_settings = admin_kernel.get_prompt_execution_settings_from_service_id("default")
admin_settings.function_choice_behavior = FunctionChoiceBehavior.Auto()

# D?finition des 3 agents conversationnels
coder_agent = ChatCompletionAgent(
    kernel=coder_kernel,
    name="CoderAgent",
    instructions=(
        "Vous ?tes le **Coder**. Votre r?le : Impl?menter avec le plus grand soin dans les cellules de code (pas la markdown) les instructions ?tablies dans les cellules de markdown du notebook\n"
        "1) Visualisez syst?matiquement le JSON du notebook via get_notebook_content().\n"
        "2) Analyser et modifier les cellules de code (pas celles de markdown) ? l'aide de la fonction update_cell_by_content(). Faites attention ? ne pas reprendre le markdown dans les cellules de code que vous ?ditez.\n"
        "3) Pr?server les marqueurs importants des cellules de code pour pouvoir les red?signer facilement lors de futures ?ditions.\n"
        "4) Quand c'est pr?t, appelez finish_implementation().\n"
        "5) Si besoin, faire de nouvelles it?rations.\n"
    ),
    arguments=KernelArguments(settings=coder_settings)
)

reviewer_agent = ChatCompletionAgent(
    kernel=reviewer_kernel,
    name="ReviewerAgent",
    instructions=(
        "Vous ?tes le **Reviewer**. Votre r?le: v?rifier avec le plus grand soin le travail du codeur, l'aiguiller pour d'?ventuelles corrections, et valider son travail quand il est parfait.\n"
        "1) Commencez toujours par consulter le notebook complet (get_notebook_content()) apr?s chaque mise ? jour.\n"
        "2) Appelez validate_notebook(approve=True/False) selon la qualit?.\n"
        "   - True => passage ? 'tested'\n"
        "   - False => retour ? 'specified'\n"
        "3) V?rifiez scrupuleusement les ?ventuelles erreurs et la bonne impl?mentation dans les cellules de code des notions pr?sent?es dans les cellules de Markdown.\n"
    ),
    arguments=KernelArguments(settings=reviewer_settings)
)

admin_agent = ChatCompletionAgent(
    kernel=admin_kernel,
    name="AdminAgent",
    instructions=(
        "Vous ?tes l'**Admin**. Vous ?tes en charge de la r?alisation du notebook et vous faites preuve d'un grand niveau d'exigence avec votre ?quipe pour assurer un rendu de la plus grande qualit?.\n"
        "1) **Vous intervenez au d?but** de la conversation pour sp?cifier au mieux le notebook: il vous appartient de le d?velopper pour laisser le minimum d'interpr?tation possible ? votre ?quipe de d?veloppeur.\n"
        "Corrigez, reformulez, D?taillez les explications dans toutes les cellules de Markdown (pas le code) o? c'est possible de sorte que les cellules de code pourront ?tre impl?ment?es, corrig?es ou finalis?es sans ambigu?t?. Vous pouvez ?diter les cellules de markdown en utilisant la fonction admin_edit_cell_by_content. Les explications fournies dans le Markdown doivent ?tre didactiques, d?taill?es et limpides. \n"
        "2) Apr?s chaque ?dition de votre part, l'agent codeur est solicit? pour l'impl?mentation des cellules de code.\n"
        "3) Quand l'agent reviewer a valid? les ?ditions du codeur, vous pouvez ? nouveau l'?diter ou le valider ou l'invalider via la fonction approve_notebook(admin_ok=True/False). Soyez intransigeant et n'h?sitez pas ? d?dire le reviewer en affinant les instructions pour le codeur tant que le r?sultat n'est pas de tr?s haute qualit?.\n"
        "4) Une fois satisfait, validez d?finitivement (admin_ok=True) pour passer l'?tat ? 'validated'.\n"
    ),
    arguments=KernelArguments(settings=admin_settings)
)



# Mettre en place la strat?gie de s?lection & la strat?gie d'arr?t
termination_strategy = ApprovedBasedTerminationStrategy(notebook_state)
selection_strategy = NotebookAwareSelectionStrategy(notebook_state)

# GroupChat pour orchestrer la conversation
group_chat = AgentGroupChat(
    agents=[coder_agent, reviewer_agent, admin_agent],
    selection_strategy=selection_strategy,
    termination_strategy=termination_strategy
)

logger.info("Agents cr??s et group_chat initialis? avec instructions mises ? jour.")


## 8. Boucle de conversation

Nous lan?ons enfin la conversation multi-agents :

- ? chaque it?ration, l?agent s?lectionn? d?pend de l??tat (`specified`, `implemented`, `tested`, `validated`).  
- Les agents peuvent appeler leurs plugins (ex. : `update_cell_by_content`, `validate_notebook`, `approve_notebook`, etc.).  
- La conversation s?arr?te d?s que le notebook est valid? ou si le quota d?it?rations est d?pass?.

Le statut final du notebook (approuv? ou non) est alors visible dans les logs et dans son contenu.


In [ ]:
import asyncio

async def run_conversation():
    """
    Lance la conversation multi-agents pour finaliser le notebook.
    S'arr?te si le notebook est valid? ou si on d?passe un max d'it?rations.
    """
    try:
        logger.info("Version initiale du notebook :")
        notebook_state.log_notebook_state()

        # Contexte initial pour la conversation
        initial_content = notebook_state.get_notebook_json()
        group_chat.history.add_system_message(f"NOTEBOOK CONTENT:\n{initial_content}")
        group_chat.history.add_user_message("Bonjour, merci de finaliser ce notebook.")

        logger.info("=== D?but de la conversation entre agents ===")
        iteration = 0

        async for message in group_chat.invoke():
            iteration += 1
            role = message.name
            content = message.content
            logger.info(f"[STEP {iteration} - {role}] {content}")

            # V?rifier si le notebook est d?j? valid?
            if notebook_state.is_approved():
                logger.info("Notebook approuv? => fin de la conversation.")
                break

        logger.info("Version finale du notebook, apr?s la conversation :")
        notebook_state.log_notebook_state(max_length=20000)

    except Exception as e:
        logger.error(f"Erreur inattendue: {str(e)}")
    finally:
        logger.info(f"Statut final - Approuv?: {notebook_state.is_approved()}")
        logger.info("=== Fin de la conversation ===")


# Ex?cuter la conversation
await run_conversation()


### Exercice 3 : Analyse de la conversation multi-agents

La conversation entre les 3 agents produit une serie de messages. Analyser ces echanges permet de comprendre le comportement du syst?me.

**Objectif** : Implementer une fonction `analyze_conversation()` qui extrait des statistiques de la conversation des agents.

**Indices** :
- # ?tape 1 : La conversation est stockee dans `group_chat.history` (ChatHistory SK)
- # ?tape 2 : Compter le nombre de messages par agent (CoderAgent, ReviewerAgent, AdminAgent)
- # ?tape 3 : Calculer la longueur moyenne des messages et le ratio messages/it?rations
- # Indice : Chaque message a un attribut `name` (nom de l'agent) et `content` (texte)

In [ ]:
def analyze_conversation(group_chat_instance) -> dict:
    """
    TODO etudiant : Analyser les statistiques de la conversation multi-agents.
    
    Returns:
        dict avec les cles: total_messages, messages_per_agent (dict),
        avg_message_length, iterations_used
    """
    # TODO etudiant : parcourir l'historique et calculer les stats
    return None

# Pour tester apres execution de la conversation :
# stats = analyze_conversation(group_chat)
# print(f"Messages par agent: {stats['messages_per_agent']}")
print("Exercice a completer")